In [ ]:
from helper_functions import *
from rapidfuzz import process, fuzz
import os
from pathlib import Path
import pandas as pd

current_dir = Path.cwd().resolve().parent
data_dir = current_dir / 'data'

try:
    compustat_df = pd.read_csv(data_dir / 'CompustatNames.csv')
    cik_df = pd.read_csv(data_dir / 'CIK.csv')
    fdic_df = pd.read_csv(data_dir / 'FDIC_clean.csv') # Using your 'FDIC_clean.csv'
    sec_df = pd.read_csv(data_dir / 'SEC_Institutions.csv')
except FileNotFoundError as e:
    print(f"Error loading file: {e}")
    print(f"Please make sure your CSV files are in the directory: {data_dir}")
    exit()
print("Finished Reading Files...")

# -----------------------------------------------------------------
# NAME CLEANING
# -----------------------------------------------------------------
print("Cleaning file names for standardized name matching...")
# cleaning and standardizing organization names
compustat_df['std_name'] = compustat_df['conm'].apply(to_standardized_name)
fdic_df['std_name'] = fdic_df['NAME'].apply(to_standardized_name)
cik_df['std_name'] = cik_df['company_name'].apply(to_standardized_name)
sec_df['std_name'] = sec_df['Name'].apply(to_standardized_name)
print("Done creating standardized names")

print("Cleaning file names for R package regular expression matching...")
# cleaning and standardizing organization names
compustat_df['clean_alias'] = compustat_df['conm'].apply(clean_org_alias)
fdic_df['clean_alias'] = fdic_df['NAME'].apply(clean_org_alias)
cik_df['clean_alias'] = cik_df['company_name'].apply(clean_org_alias)
print("Done creating cleaned alias names")


duplicates_cik_id = cik_df['cik'].duplicated(keep = False)
duplicates_cik = cik_df[duplicates_cik_id]

# This is the final dataframe crosswalk where we will be merging entities into. 
cols = ['aliases', 'standardized_names', 'clean_alias', 'cik', 'FED_RSSD', 'ticker', 'naics', 'sources', 
        'matching_type', 'fuzzy_matching_score']
final_crosswalk_df = pd.DataFrame(columns = cols)

# -----------------------------------------------------------------
# CIK MATCHING
# -----------------------------------------------------------------
# Merging entities in cik_df based on the unique cik id value. 
print("Now merging entities in cik_df based on the unique cik id value.")
grouped_by_cik_id = cik_df.groupby('cik')
confident_matches = []
# count = 0
for cik_value, group in grouped_by_cik_id:
    # count += 1
    # if count % 1000 == 0:
    #   print(count)
    
    if len(group) > 1:
        # Aggregate the data based on cik
        new_match_keys = {
            'cik': group['cik'].dropna().unique().tolist(),
            # Now aggregate the std_name and clean_alias to see all variations found for cik
            'standardized_names': '|'.join(group['std_name'].dropna().unique()),
            'clean_alias': '|'.join(group['clean_alias'].dropna().unique()),
            # Aggregate other fields as before
            'aliases': '|'.join(group['company_name'].dropna().unique()),
            'sources': 'cik',
            'matching_type': 'cik_id_match'
        }
        confident_matches.append(new_match_keys)
    else: 
        unmatched_keys = {
            'cik': group['cik'].dropna().unique().tolist(),
            'standardized_names': group['std_name'].iloc[0],
            'clean_alias': group['clean_alias'].iloc[0],
            'aliases': group['company_name'].iloc[0],
            'sources': 'cik'
        }
        confident_matches.append(unmatched_keys)
         
pd.set_option('display.max_colwidth', None)
enriched_cik_df = pd.DataFrame(confident_matches)
print(f"cik_df reduced to {len(enriched_cik_df)} entities after merging.")

final_crosswalk_df = pd.concat([final_crosswalk_df, enriched_cik_df])
print("Added enriched cik_df to the final_crosswalk_df")

temp_compustat_df = compustat_df[compustat_df['cik'].notna()].copy()

# Apply the cleaning function to both dataframes
temp_compustat_df['cik'] = temp_compustat_df['cik'].astype('string')
temp_compustat_df.loc[:, 'cik'] = temp_compustat_df['cik'].apply(clean_cik)
final_crosswalk_df.loc[:, 'cik'] = final_crosswalk_df['cik'].apply(clean_cik)

# Merge compustat_df into final_crosswalk_df based on cik id. 
final_crosswalk_df = final_crosswalk_df.merge(temp_compustat_df, on = 'cik', how = 'left', 
                                              suffixes=('','_other'), indicator=True)
print("Successfully merged compustat into final_crosswalk_df based on cik id")


In [ ]:
CIK_merge_cleaup(final_crosswalk_df, "conm", "compustat", "tic", naics_column_name = "naics")

In [ ]:
final_crosswalk_df = final_crosswalk_df.drop(columns=['Unnamed: 0','gvkey', 'conm', 'cusip', 'sic', 'tic',\
                                                      'gsubind', 'gind', 'year1', 'year2', 'std_name', 
                                                      '_merge', 'clean_alias_other', 'naics_other'])


In [ ]:
# Merge sec_df into final_crosswalk_df based on cik id. 
sec_df = sec_df.rename(columns = {'CIK': 'cik'})
sec_df.loc[:, 'cik'] = sec_df['cik'].apply(clean_cik)
final_crosswalk_df = final_crosswalk_df.merge(sec_df, on = 'cik', how = 'left', suffixes=('','_other'), indicator=True)

In [ ]:
final_crosswalk_df

In [ ]:
CIK_merge_cleaup(final_crosswalk_df, "Name", "sec", "Ticker")

In [ ]:
final_crosswalk_df = final_crosswalk_df.drop(columns=['Ticker', 'index', 'Name', 'Exchange', 'SIC', 
                                                      'Business', 'Incorporated', 'IRS', '_merge', 'std_name'])

In [ ]:
final_crosswalk_df